In [8]:
import fitz #PyMuPDF for reading PDF
from langchain_core.documents import documents
from transformers import CLIPProcessor, CLIPModel #used for converting text and images into vector embeddings, it is an opensource model trained on images and text. It is a combination of vision transformers and transformers (text transformers are called as transformers)
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.schema.messages import HumanMessage
from sklearn.metrics.pairwise import cosin_similarity
import os
import base64
import io 
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

ModuleNotFoundError: No module named 'fitz'

In [ ]:
###CLip Model

import os
from dotenv import load_dotenv
load_dotenv()

#setting up the environment
os.environ["OPENAI_API_KEY"]=os.getenv("OPEN_API_KEY")

### initialize the CLIP model for unified embeddings
#variable clip_model, the model is called from hugging face. the model name is openai/clip-vit-base-patch32
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
#clip_processor is the variable. It converts the input into the format the model requires
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model_eval()

In [ ]:
###Embedding Functions
def embed_image(image_data):
    """Embed image using CLIP"""
    if isinstance(image_data, str): #if Path
       image = Image.open(image_data).convert("RGB")
    else: #if PIL Image
        image = image_data
    #converting the format into tensors
    inputs = clip_processor(image=image,return_tensors="pt")
    with torch.no_grad():
        #built-in feature of CLIP model that will get the image features from the image input
        features = clip_model.get_image_features(**inputs)
        #normalize embeddings to unit vector
        features = features / features.norm(dim = -1, keepdim=True)
        return features.squeeze().numpy()

def embed_text(text):
    """Embed text using CLIP"""
    inputs = clip_processor
    (text=text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=77 #CLIP's max token length 
    )
    with torch.no_grad():
    features = clip_model.get_text_features(**inputs)
    #normalize embeddings
    features = features / features.norm(dim=-1, keepdim=True)
    return features.squeeze().numpy()

In [ ]:
### Process PDF
pdf_path="multimodal_sample.pdf"
doc=fitz.open(pdf_path)
# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {} #store actual image data for the LLM

#Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [ ]:
for i,page in enumerate(doc):
    ##process text
    text=page.get_text()
    #remove empty spaces
    if text.strip():
        ##create temporary documents for splitting
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"}) #for all the text content keep the metadata type as text
        text_chunks = splitter.split_documents([temp_doc])

        #Embed each chunk using CLI
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content) #calls the embed_text function, which will convert to vector and normalize the vector
            all_embeddings.append(embedding)
            all_doc.append(chunk)
